# Data Cleaning  - NHL Play-by-Play 

this notebook shows how to clean the nhl data from the api...



In [1]:
# imports
import pandas as pd
import numpy as np

import sys
sys.path.append('..')

from ift6758.data import SeasonData
from ift6758.data.data_cleaning import clean_season_data, get_additional_features, clean_play_by_play_data


pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Load the data

first need to get the raw data 

In [2]:
# lets get 2016 season
print("getting data for 2016...")
season_2016 = SeasonData(2016)
season_2016.get_data_from_api()


print(f"reg season: {len(season_2016.reg_season_data)}")
print(f"playoffs: {len(season_2016.playoffs_data)}")

getting data for 2016...
reg season: 1230
playoffs: 87


##  Clean one game first



In [3]:
# Reload the module to get the updated function
import importlib
from ift6758.data import data_cleaning
importlib.reload(data_cleaning)
from ift6758.data.data_cleaning import clean_play_by_play_data

if season_2016.reg_season_data:
    # just get first game
    sample_game_id = list(season_2016.reg_season_data.keys())[0]
    sample_game = season_2016.reg_season_data[sample_game_id]
    
    cleaned_game = clean_play_by_play_data(sample_game)
    
    print(f"Game id: {sample_game_id}")
    print(f"Events found: {len(cleaned_game)}")
    print("\n")
    print(cleaned_game.head(10))
else:
    print("no data...run collection first")

Game id: 2016020001
Events found: 68


      game_id  period period_time  team event_type  x_coord  y_coord  \
0  2016020001       1       01:11    10       Shot    -77.0      5.0   
1  2016020001       1       02:53     9       Shot     86.0     13.0   
2  2016020001       1       04:01     9       Shot     23.0    -38.0   
3  2016020001       1       04:46     9       Shot     33.0    -15.0   
4  2016020001       1       06:46    10       Shot    -34.0     28.0   
5  2016020001       1       07:30    10       Shot    -33.0    -17.0   
6  2016020001       1       08:21    10       Goal    -70.0      1.0   
7  2016020001       1       08:29    10       Shot    -45.0    -36.0   
8  2016020001       1       09:00     9       Shot     33.0    -18.0   
9  2016020001       1       10:16     9       Shot     34.0     20.0   

   standardized_x_coord  standardized_y_coord         shooter          goalie  \
0                  77.0                  -5.0  Player_8478483  Goalie_8467950   
1     

## now clean whole season

In [4]:
# Reload the module to get the updated function
import importlib
from ift6758.data import data_cleaning
importlib.reload(data_cleaning)
from ift6758.data.data_cleaning import clean_season_data

# clean everything
if season_2016.reg_season_data or season_2016.playoffs_data:
    print("cleaning 2016-17...")
    cleaned_season_df = clean_season_data(season_2016)
    
    
    print(f"\ntotal: {len(cleaned_season_df)}")
    
    print("\nsample:")
    print(cleaned_season_df.head(10))

cleaning 2016-17...

total: 80399

sample:
      game_id  period period_time  team event_type  x_coord  y_coord  \
0  2016020001       1       01:11    10       Shot    -77.0      5.0   
1  2016020001       1       02:53     9       Shot     86.0     13.0   
2  2016020001       1       04:01     9       Shot     23.0    -38.0   
3  2016020001       1       04:46     9       Shot     33.0    -15.0   
4  2016020001       1       06:46    10       Shot    -34.0     28.0   
5  2016020001       1       07:30    10       Shot    -33.0    -17.0   
6  2016020001       1       08:21    10       Goal    -70.0      1.0   
7  2016020001       1       08:29    10       Shot    -45.0    -36.0   
8  2016020001       1       09:00     9       Shot     33.0    -18.0   
9  2016020001       1       10:16     9       Shot     34.0     20.0   

   standardized_x_coord  standardized_y_coord         shooter          goalie  \
0                  77.0                  -5.0  Player_8478483  Goalie_8467950   
1 

In [8]:

# add more features
if 'cleaned_season_df' in locals():
    enhanced_df = get_additional_features(cleaned_season_df)
    
    print("added features like distance, angle etc")
    
    print("\n")
    print(enhanced_df[['game_id', 'period', 'event_type', 'distance_from_net', 'angle_from_net']].head())

added features like distance, angle etc


      game_id  period event_type  distance_from_net  angle_from_net
0  2016020001       1       Shot          13.000000       22.619865
1  2016020001       1       Shot          13.341664       77.005383
2  2016020001       1       Shot          76.157731       29.931512
3  2016020001       1       Shot          57.974132       14.995079
4  2016020001       1       Shot          61.717096       26.980231


### check the data quality



In [9]:
if 'enhanced_df' in locals():
    # stats
    print("Data stats:")
    print(f"events: {len(enhanced_df)}")
    print(f"goals: {(enhanced_df['event_type'] == 'Goal').sum()}")
    print(f"shots: {(enhanced_df['event_type'] == 'Shot').sum()}")
    
    shooting_pct = (enhanced_df['event_type'] == 'Goal').mean() * 100
    print(f"shooting %: {shooting_pct:.2f}%")
    
    
    print("\nmissing:")
    print(enhanced_df.isnull().sum())
    
    print("\n")
    print("event types:")
    print(enhanced_df['event_type'].value_counts())
    
    print("\nstrength:")
    print(enhanced_df['strength'].value_counts())
    

Data stats:
events: 80399
goals: 7377
shots: 73022
shooting %: 9.18%

missing:
game_id                   0
period                    0
period_time               0
team                      0
event_type                0
x_coord                   4
y_coord                   4
standardized_x_coord      4
standardized_y_coord      4
shooter                   0
goalie                  314
shot_type                 0
empty_net                 0
strength                  0
season                    0
game_type                 0
distance_from_net         4
angle_from_net            4
period_seconds            0
game_seconds              0
is_overtime               0
dtype: int64


event types:
event_type
Shot    73022
Goal     7377
Name: count, dtype: int64

strength:
strength
Even    80399
Name: count, dtype: int64


In [10]:

# get multiple years
seasons_to_process = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
all_seasons_data = []


for year in seasons_to_process:
    print(f"\ngetting {year}...")
    
    season_obj = SeasonData(year)
    season_obj.get_data_from_api()
    
    cleaned_df = clean_season_data(season_obj)
    
    if not cleaned_df.empty:
        cleaned_df = get_additional_features(cleaned_df)
        all_seasons_data.append(cleaned_df)
        
        print(f"  events: {len(cleaned_df)}")
        print(f"  goals: {(cleaned_df['event_type'] == 'Goal').sum()}")

# combine
if all_seasons_data:
    combined_df = pd.concat(all_seasons_data, ignore_index=True)
    print(f"\n\ntotal all seasons: {len(combined_df)}")
    print(f"games: {combined_df['game_id'].nunique()}")


getting 2016...
  events: 80399
  goals: 7377

getting 2017...
  events: 87137
  goals: 8187

getting 2018...
  events: 85939
  goals: 8250

getting 2019...
  events: 73867
  goals: 7122

getting 2020...
  events: 57734
  goals: 5636

getting 2021...
  events: 84334
  goals: 8368

getting 2022...
  events: 88086
  goals: 9027

getting 2023...
  events: 84932
  goals: 8772


total all seasons: 642428
games: 10269


In [11]:

# for blog post
if 'enhanced_df' in locals():
    print("cleaned data sample:")
    display_cols = ['game_id', 'period', 'period_time', 'team', 'event_type', 
                   'x_coord', 'y_coord', 'shooter', 'goalie', 'shot_type', 
                   'empty_net', 'strength']
    print(enhanced_df[display_cols].head(10))

cleaned data sample:
      game_id  period period_time  team event_type  x_coord  y_coord  \
0  2016020001       1       01:11    10       Shot    -77.0      5.0   
1  2016020001       1       02:53     9       Shot     86.0     13.0   
2  2016020001       1       04:01     9       Shot     23.0    -38.0   
3  2016020001       1       04:46     9       Shot     33.0    -15.0   
4  2016020001       1       06:46    10       Shot    -34.0     28.0   
5  2016020001       1       07:30    10       Shot    -33.0    -17.0   
6  2016020001       1       08:21    10       Goal    -70.0      1.0   
7  2016020001       1       08:29    10       Shot    -45.0    -36.0   
8  2016020001       1       09:00     9       Shot     33.0    -18.0   
9  2016020001       1       10:16     9       Shot     34.0     20.0   

          shooter          goalie shot_type  empty_net strength  
0  Player_8478483  Goalie_8467950     wrist      False     Even  
1  Player_8467967  Goalie_8475883     wrist      False



done - data is cleaned and ready for viz